# Preparación de datos para ancestría local (cromosoma 22)

**Este notebook lo corre la instructora una sola vez; los participantes no lo usan.**

Descarga el panel HGDP + 1000 Genomes ya faseado con SHAPEIT5 (Koenig et al., 2024), selecciona las poblaciones del curso y genera archivos pequeños para la práctica de ancestría local (semana 5). Tarda unos 10–15 minutos y al final descarga `datos_ancestria_local.zip`.

| Rol | Poblaciones |
|---|---|
| Mezcladas (consulta) | CLM (Medellín, Colombia), MXL (Los Ángeles, ascendencia mexicana), PUR (Puerto Rico), PEL (Lima, Perú) |
| Referencia africana | YRI (Ibadán, Nigeria) |
| Referencia europea | IBS (España) |
| Referencia nativa americana | Karitiana, Surui, Pima y Colombian (HGDP) |

Se usan YRI e IBS porque la ancestría africana de las poblaciones latinoamericanas proviene sobre todo de África occidental, y la europea sobre todo de la península ibérica (Montinaro et al., 2015; Homburger et al., 2015).

No se usan los Maya de HGDP como referencia nativa americana: en una prueba con RFMix sobre el cromosoma 22, 8 de 21 tenían más de 5 % de ancestría no nativa (uno, 41 % europea). Una referencia mezclada haría que RFMix confundiera segmentos europeos o africanos con nativos americanos.

## 1. Instalar bcftools

In [ ]:
!apt-get -qq install -y bcftools > /dev/null
!bcftools --version | head -1

## 2. Descargar los datos

- El cromosoma 22 faseado (279 MB) y su índice.
- La tabla de metadatos, que dice a qué población pertenece cada muestra.

In [ ]:
!wget -nv -nc https://storage.googleapis.com/gcp-public-data--gnomad/resources/hgdp_1kg/phased_haplotypes_v2/hgdp1kgp_chr22.filtered.SNV_INDEL.phased.shapeit5.bcf
!wget -nv -nc https://storage.googleapis.com/gcp-public-data--gnomad/resources/hgdp_1kg/phased_haplotypes_v2/hgdp1kgp_chr22.filtered.SNV_INDEL.phased.shapeit5.bcf.csi
!wget -nv -nc https://storage.googleapis.com/gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg_v2/metadata_and_qc/gnomad_meta_updated.tsv
!ls -lh hgdp1kgp_chr22.filtered.SNV_INDEL.phased.shapeit5.bcf* gnomad_meta_updated.tsv

## 3. Elegir las muestras

Primero, la lista de muestras que hay en el archivo del cromosoma 22.

In [ ]:
!bcftools query -l hgdp1kgp_chr22.filtered.SNV_INDEL.phased.shapeit5.bcf > muestras_en_bcf.txt
!wc -l muestras_en_bcf.txt

Luego cruzamos con los metadatos. Se excluyen las muestras marcadas como emparentadas con otra (`sample_filters.release_related`), para no repetir haplotipos casi idénticos.

**Excepción:** las referencias nativas americanas de HGDP. Son poblaciones pequeñas y aisladas, con muchos parientes entre sí: con el filtro quedarían solo 2 Karitiana y 1 Surui. En una referencia importa más tener la mayor diversidad de haplotipos posible, así que en ellas no se aplica el filtro.

In [ ]:
import pandas as pd

consulta = ['CLM', 'MXL', 'PUR', 'PEL']
referencia = {'YRI': 'AFR', 'IBS': 'EUR',
              'Karitiana': 'NAT', 'Surui': 'NAT', 'Pima': 'NAT', 'Colombian': 'NAT'}

cols = {'s': 'muestra', 'hgdp_tgp_meta.Population': 'poblacion',
        'hgdp_tgp_meta.Project': 'proyecto', 'sample_filters.release_related': 'emparentada'}
meta = pd.read_csv('gnomad_meta_updated.tsv', sep='\t', usecols=list(cols), dtype=str).rename(columns=cols)

en_bcf = set(open('muestras_en_bcf.txt').read().split())
sel = meta[meta['poblacion'].isin(consulta + list(referencia)) & meta['muestra'].isin(en_bcf)].copy()
sel['emparentada'] = sel['emparentada'].str.lower().eq('true')

nativas = [p for p, a in referencia.items() if a == 'NAT']
sel['excluir'] = sel['emparentada'] & ~sel['poblacion'].isin(nativas)

resumen = sel.groupby('poblacion').agg(en_el_archivo=('muestra', 'size'),
                                       emparentadas=('emparentada', 'sum'),
                                       excluidas=('excluir', 'sum'))
resumen['se_usan'] = resumen['en_el_archivo'] - resumen['excluidas']
resumen

Guardamos las listas de muestras y el *sample map* de RFMix (muestra de referencia → ancestría).

In [ ]:
sel = sel[~sel['excluir']].copy()
sel['rol'] = sel['poblacion'].map(lambda p: 'consulta' if p in consulta else 'referencia')
sel['ancestria'] = sel['poblacion'].map(referencia)

sel.loc[sel['rol'] == 'consulta', 'muestra'].to_csv('muestras_consulta.txt', index=False, header=False)
sel.loc[sel['rol'] == 'referencia', 'muestra'].to_csv('muestras_referencia.txt', index=False, header=False)
sel.loc[sel['rol'] == 'referencia', ['muestra', 'ancestria']].to_csv('sample_map.txt', sep='\t', index=False, header=False)
sel[['muestra', 'poblacion', 'proyecto', 'rol', 'ancestria']].to_csv('poblaciones.tsv', sep='\t', index=False)

sel.groupby(['rol', 'ancestria'], dropna=False).size()

## 4. Filtrar las variantes y separar consulta y referencia

Se conservan solo SNPs bialélicos con frecuencia del alelo menor ≥ 5 % en las muestras elegidas, y solo el genotipo faseado (`GT`). Así los archivos quedan pequeños para Colab y GitHub.

In [ ]:
!cat muestras_consulta.txt muestras_referencia.txt > muestras_todas.txt
!bcftools view --threads 2 --force-samples -S muestras_todas.txt -Ou hgdp1kgp_chr22.filtered.SNV_INDEL.phased.shapeit5.bcf \
  | bcftools view -m2 -M2 -v snps -q 0.05:minor -Ou \
  | bcftools annotate -x INFO,^FORMAT/GT -Oz -o chr22_seleccion.vcf.gz
!bcftools index -t chr22_seleccion.vcf.gz

In [ ]:
!bcftools view -I -S muestras_consulta.txt -Oz -o chr22_consulta.vcf.gz chr22_seleccion.vcf.gz
!bcftools view -I -S muestras_referencia.txt -Oz -o chr22_referencia.vcf.gz chr22_seleccion.vcf.gz
!bcftools index -t chr22_consulta.vcf.gz
!bcftools index -t chr22_referencia.vcf.gz

Revisión: número de muestras y de SNPs en cada archivo, y que los genotipos estén faseados (separados por `|`).

In [ ]:
%%bash
for f in chr22_consulta.vcf.gz chr22_referencia.vcf.gz; do
  echo "$f: $(bcftools query -l $f | wc -l) muestras, $(bcftools index -n $f) SNPs"
done
bcftools view -H chr22_consulta.vcf.gz | head -2 | cut -f1-12

## 5. Mapa genético del cromosoma 22 (GRCh38)

RFMix necesita saber la distancia genética (en cM) entre los SNPs. Usamos el mapa en GRCh38 que distribuye Eagle (derivado del mapa de HapMap fase II) y lo convertimos al formato de RFMix: cromosoma, posición en pb y posición en cM, separados por tabuladores.

In [ ]:
!wget -nv -nc https://storage.googleapis.com/broad-alkesgroup-public/Eagle/downloads/tables/genetic_map_hg38_withX.txt.gz

In [ ]:
%%bash
echo "Archivo original:"
zcat genetic_map_hg38_withX.txt.gz | head -3
zcat genetic_map_hg38_withX.txt.gz \
  | awk -v OFS='\t' 'NR > 1 && ($1 == "22" || $1 == "chr22") {print "chr22", $2, $4}' > genetic_map_chr22_GRCh38.txt
echo "Formato RFMix:"
head -3 genetic_map_chr22_GRCh38.txt
wc -l genetic_map_chr22_GRCh38.txt

## 6. Grupo para la clase

RFMix tarda varios minutos, así que en la clase se analiza en vivo un grupo pequeño: 10 personas de CLM elegidas al azar (con semilla fija, para que siempre sean las mismas).

In [ ]:
clase = sel[sel['poblacion'] == 'CLM'].sample(10, random_state=2026)['muestra'].sort_values()
clase.to_csv('muestras_clase.txt', index=False, header=False)
print(', '.join(clase))

In [ ]:
!bcftools view -I -S muestras_clase.txt -Oz -o chr22_consulta_clase.vcf.gz chr22_consulta.vcf.gz
!bcftools query -l chr22_consulta_clase.vcf.gz | wc -l

## 7. Empaquetar y descargar

In [ ]:
!zip -j datos_ancestria_local.zip chr22_consulta.vcf.gz chr22_consulta.vcf.gz.tbi chr22_consulta_clase.vcf.gz chr22_referencia.vcf.gz chr22_referencia.vcf.gz.tbi sample_map.txt poblaciones.tsv genetic_map_chr22_GRCh38.txt
!ls -lh datos_ancestria_local.zip

In [ ]:
from google.colab import files
files.download('datos_ancestria_local.zip')

| Archivo del zip | Qué es |
|---|---|
| `chr22_consulta.vcf.gz` (+ `.tbi`) | Haplotipos faseados de las poblaciones mezcladas (CLM, MXL, PUR, PEL) |
| `chr22_consulta_clase.vcf.gz` | Las 10 personas de CLM que se analizan en vivo en la clase |
| `chr22_referencia.vcf.gz` (+ `.tbi`) | Haplotipos faseados de las poblaciones de referencia |
| `sample_map.txt` | Para RFMix: cada muestra de referencia y su ancestría (AFR, EUR o NAT) |
| `poblaciones.tsv` | Población, proyecto y rol de cada muestra (para los gráficos) |
| `genetic_map_chr22_GRCh38.txt` | Mapa genético del cromosoma 22 en formato RFMix |

Los `.tbi` son los índices de los VCF y permiten leerlos más rápido.

## Referencias

- Koenig Z, et al. A harmonized public resource of deeply sequenced diverse human genomes. *Genome Res*. 2024;34(5):796. doi:10.1101/gr.278378.123
- Byrska-Bishop M, et al. High-coverage whole-genome sequencing of the expanded 1000 Genomes Project cohort including 602 trios. *Cell*. 2022;185:3426–3440. doi:10.1016/j.cell.2022.08.004
- Bergström A, et al. Insights into human genetic variation and population history from 929 diverse genomes. *Science*. 2020;367:eaay5012. doi:10.1126/science.aay5012
- Montinaro F, et al. Unravelling the hidden ancestry of American admixed populations. *Nat Commun*. 2015;6:6596. doi:10.1038/ncomms7596
- Homburger JR, et al. Genomic insights into the ancestry and demographic history of South America. *PLoS Genet*. 2015;11:e1005602. doi:10.1371/journal.pgen.1005602
- Loh PR, et al. Reference-based phasing using the Haplotype Reference Consortium panel. *Nat Genet*. 2016;48:1443–1448. doi:10.1038/ng.3679 (Eagle, que distribuye el mapa genético en GRCh38)
- International HapMap Consortium. A second generation human haplotype map of over 3.1 million SNPs. *Nature*. 2007;449:851–861. doi:10.1038/nature06258 (origen del mapa genético)